# 1. External Data Ingestion Layer

## Retrieval Objective

The first technical layer of the project was the ingestion of raw external records from public sources.

The purpose of this layer was not simply to download numerical values. The retrieval workflow was designed to establish a repeatable entry point for the full analytical pipeline. To do that, the system had to collect heterogeneous source data, isolate source-specific access logic, and prepare raw outputs in a form that could later be standardized into a reusable internal schema.

This made retrieval the beginning of the analytical workflow rather than a separate utility step. Without a stable ingestion layer, downstream structuring, metric engineering, and output delivery would remain manual, inconsistent, or difficult to reproduce.

## Source Integration Strategy

The ingestion layer combined two external public sources that represented different parts of the analytical problem.

The first source provided accounting-based annual records. These records formed the basis for reconstructing normalized multi-year snapshots. The second source provided current market price data, which extended the workflow beyond reporting data alone and made it possible to attach price-based analytical context to the latest annual record.

The technical significance of this design was not limited to the financial domain itself. A single-source workflow would have been simpler, but it would also have been less representative of real analytical environments, where useful outputs often depend on integrating records from sources with different structures, update cycles, and semantic roles.

By combining both sources in the same retrieval layer, the project established an input architecture that could support downstream normalization, transformation, and interpretation within one connected workflow.

In [ ]:
def resolve_cik_from_ticker(ticker: str) -> Optional[str]:

    ticker = _normalize_ticker(ticker)
    data = _safe_get_json(TICKER_CIK_URL, headers=SEC_HEADERS)
    if not data:
        return None

    for _, row in data.items():
        if str(row.get("ticker", "")).upper() == ticker:
            cik_int = row.get("cik_str")
            if cik_int is None:
                return None
            return str(cik_int).zfill(10)

    return None

In [ ]:
def get_latest_eod_price(symbol: str, api_key: str) -> Optional[Dict[str, Any]]:

    symbol = _normalize_symbol(symbol)

    if not symbol or not api_key:
        return None

    params = {
        "access_key": api_key,
        "symbols": symbol,
        "limit": 1,
        "sort": "DESC",
    }

    data = _safe_get(BASE_URL, params=params, timeout=10)
    if not data:
        return None

    if "error" in data:
        return None

    records = data.get("data") or []
    if not records:
        return None

    latest = records[0]

    return {
        "symbol": latest.get("symbol"),
        "date": latest.get("date"),
        "open": latest.get("open"),
        "high": latest.get("high"),
        "low": latest.get("low"),
        "close": latest.get("close"),
        "adj_close": latest.get("adj_close"),
        "volume": latest.get("volume"),
        "exchange": latest.get("exchange"),
    }

## Retrieval Workflow

The retrieval process was designed as a staged workflow.

First, the application accepts a ticker as the entry key.  
Second, the ticker is resolved into the identifier needed to access the accounting-based external source.  
Third, multi-year annual records are collected from the filing source.  
Fourth, current market price data is retrieved separately from the pricing source.  
Finally, both outputs are passed downstream in minimal structured forms so that later layers can focus on schema construction, metric engineering, and delivery logic rather than on source-specific retrieval details.

This staged design improved traceability across the pipeline. It also made it easier to separate retrieval issues from later problems in transformation or interpretation.

In [ ]:
def fetch_10k_snapshots(
    ticker: str,
    count: int = 5,
    current_year: Optional[int] = None,
) -> List[FinancialSnapshot]:

    if current_year is None:
        current_year = dt.datetime.now().year

    snapshots: List[FinancialSnapshot] = []

    for i in range(count):
        year = current_year - i
        snapshot = fetch_single_10k(ticker, year)
        if snapshot:
            snapshots.append(snapshot)

    return sorted(snapshots, key=lambda x: x.fiscal_year)

## Why the Retrieval Layer Matters

The retrieval layer mattered because it defined the boundary between raw external records and the rest of the analytical system.

From a technical perspective, this layer demonstrated more than simple API usage. It showed how source-specific access logic can be isolated from downstream processing, how multiple external inputs can be integrated into one workflow, and how raw responses can be prepared for later normalization without manual intervention.

From a data perspective, the layer was important for three reasons.

First, it connected the project to real external source data rather than pre-cleaned local tables.  
Second, it determined which variables could later be standardized and transformed into reusable metrics.  
Third, it established a repeatable ingestion pattern that can be generalized to other workflows involving fragmented external records.

For that reason, the retrieval stage should be understood as the first analytical layer of the project rather than as a low-level implementation detail.

## Data Quality Perspective

One important lesson from the ingestion layer was that successful retrieval does not automatically produce analysis-ready data.

External source records may still contain field inconsistencies, incomplete coverage, period-alignment ambiguity, or structure differences that affect later interpretation. For that reason, the retrieval stage was treated not as a fully solved preprocessing step, but as the beginning of the project’s validation perspective.

This point became important later in the workflow, because apparently valid outputs could still require interpretation checks when annual alignment or source completeness was uncertain.

## Transition

Once the external ingestion layer was established, the next step was to convert raw retrieved values into a stable internal structure.

The following chapter explains how the source outputs were reorganized into standardized annual snapshots and why internal schema design was necessary before reusable metric engineering could begin.